In [11]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
import os


In [12]:
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
model_name = os.getenv("GROQ_MODEL")

In [13]:
model = ChatGroq(model=model_name, temperature=0) 

In [14]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [15]:
def create_outline(state:BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on this topic {title}'
    outline = model.invoke(prompt).content
    state['outline'] = outline

    return state

In [16]:
def create_blog(state:BlogState) -> BlogState:

    #fetch title
    title = state['title']

    #fetch outline
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [17]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

#edges

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()


In [18]:
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': '## Blog Outline  \n**Title:** *Rise of AI in India: From Startup Hubs to Smart Nation*  \n\n---\n\n### 1. Introduction  \n- **Hook:** A striking statistic or anecdote (e.g., “India’s AI market is projected to hit $10\u202fB by 2025”).  \n- **Purpose of the post:** Explain why the AI boom matters for India’s economy, society, and global standing.  \n- **Road‑map:** Briefly list the main sections that will be covered.\n\n---\n\n### 2. Historical Context  \n- **Early AI research in India (1970s‑1990s)**  \n  - Pioneering work at IITs, BITS, and the Indian Institute of Science.  \n  - First AI labs and government‑funded projects.  \n- **The “AI Winter” and resurgence**  \n  - Global slowdown in the 2000s.  \n  - Re‑ignition through digital transformation and mobile penetration.  \n\n---\n\n### 3. Current Landscape of AI in India  \n- **Market Size & Growth**  \n  - CAGR, projected valuation, comparison with other emerging markets.  \n- **Key Sec

In [20]:
print(final_state['outline'])

## Blog Outline  
**Title:** *Rise of AI in India: From Startup Hubs to Smart Nation*  

---

### 1. Introduction  
- **Hook:** A striking statistic or anecdote (e.g., “India’s AI market is projected to hit $10 B by 2025”).  
- **Purpose of the post:** Explain why the AI boom matters for India’s economy, society, and global standing.  
- **Road‑map:** Briefly list the main sections that will be covered.

---

### 2. Historical Context  
- **Early AI research in India (1970s‑1990s)**  
  - Pioneering work at IITs, BITS, and the Indian Institute of Science.  
  - First AI labs and government‑funded projects.  
- **The “AI Winter” and resurgence**  
  - Global slowdown in the 2000s.  
  - Re‑ignition through digital transformation and mobile penetration.  

---

### 3. Current Landscape of AI in India  
- **Market Size & Growth**  
  - CAGR, projected valuation, comparison with other emerging markets.  
- **Key Sectors Leveraging AI**  
  - **Healthcare:** diagnostics, telemedicine, drug 